# Week 2 Day 3 — Model Persistence with Joblib

## ICU Early Warning Prediction System

In Week 2 Day 2, the model predictions were converted into patient-level risk scores and clinical risk categories.

Today, the objective is to make the model reusable by saving it to disk using `joblib`.

This is important because a deployed ICU Early Warning Prediction System should not retrain the model every time it makes a prediction.

Instead, the system should:

1. train the model once
2. save the model
3. reload the model when needed
4. use it for new patient predictions

In [2]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score

from imblearn.over_sampling import SMOTE

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR = PROJECT_ROOT / "models"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)
print("Results path:", RESULTS_DIR)
print("Models path:", MODELS_DIR)
print("Models directory exists:", MODELS_DIR.exists())

Project root: c:\Users\User\OneDrive\Desktop\icu-early-warning-system
Data path: c:\Users\User\OneDrive\Desktop\icu-early-warning-system\data\processed
Results path: c:\Users\User\OneDrive\Desktop\icu-early-warning-system\results
Models path: c:\Users\User\OneDrive\Desktop\icu-early-warning-system\models
Models directory exists: True


In [4]:
data_file = DATA_PATH / "day2_patient_level_features.csv"

df = pd.read_csv(data_file)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (100, 18)


,Patient_ID,HR_mean,HR_max,HR_min,O2Sat_mean,O2Sat_min,Temp_mean,Temp_max,SBP_mean,SBP_min,MAP_mean,MAP_min,Resp_mean,Resp_max,Age_first,Gender_first,ICULOS_max,SepsisLabel_max
0,p000001,101.907407,117.0,76.0,91.453704,85.0,36.735185,37.44,127.870370,78.0,88.321111,44.00,24.555556,32.0,83.14,0,54,0
1,p000002,62.173913,94.0,54.0,97.043478,94.0,36.206087,37.00,129.043478,114.0,67.239130,50.50,14.630435,27.0,75.91,0,23,0
2,p000003,79.968750,93.0,68.0,95.375000,91.0,37.465000,38.61,139.760417,121.0,81.149167,62.67,25.302083,40.0,45.82,0,48,0
3,p000004,102.172414,113.0,89.0,98.189655,95.5,36.463103,37.00,113.017241,90.0,67.063103,34.00,18.758621,26.0,65.71,0,29,0
4,p000005,76.604167,88.0,61.0,97.677083,96.0,37.072292,37.33,135.072917,114.0,90.364583,73.00,15.447917,21.0,28.09,1,49,0


In [5]:
target_col = "SepsisLabel_max"

X = df.drop(columns=["Patient_ID", target_col])
y = df[target_col]

feature_names = X.columns.tolist()

print("Number of features:", len(feature_names))
print("Target column:", target_col)

Number of features: 16
Target column: SepsisLabel_max


In [6]:
X_train, X_test, y_train, y_test, patient_id_train, patient_id_test = train_test_split(
    X,
    y,
    df["Patient_ID"],
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (75, 16)
Testing shape: (25, 16)


In [7]:
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Training class distribution before SMOTE:")
print(y_train.value_counts())

print("\nTraining class distribution after SMOTE:")
print(y_train_smote.value_counts())

Training class distribution before SMOTE:
SepsisLabel_max
0    64
1    11
Name: count, dtype: int64

Training class distribution after SMOTE:
SepsisLabel_max
1    64
0    64
Name: count, dtype: int64


In [8]:
final_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

final_model.fit(X_train_smote, y_train_smote)

print("Final Gradient Boosting + SMOTE model trained successfully.")

Final Gradient Boosting + SMOTE model trained successfully.


In [9]:
y_pred_before = final_model.predict(X_test)
y_prob_before = final_model.predict_proba(X_test)[:, 1]

print("Classification Report Before Saving:")
print(classification_report(y_test, y_pred_before))

auc_before = roc_auc_score(y_test, y_prob_before)
print("ROC-AUC before saving:", round(auc_before, 3))

Classification Report Before Saving:
              precision    recall  f1-score   support

           0       0.95      0.82      0.88        22
           1       0.33      0.67      0.44         3

    accuracy                           0.80        25
   macro avg       0.64      0.74      0.66        25
weighted avg       0.87      0.80      0.83        25

ROC-AUC before saving: 0.758


In [10]:
model_path = MODELS_DIR / "gradient_boosting_smote_model.joblib"

joblib.dump(final_model, model_path)

print("Model saved successfully.")
print("Saved model path:", model_path)

Model saved successfully.
Saved model path: c:\Users\User\OneDrive\Desktop\icu-early-warning-system\models\gradient_boosting_smote_model.joblib


In [11]:
feature_names_path = MODELS_DIR / "model_feature_names.joblib"

joblib.dump(feature_names, feature_names_path)

print("Feature names saved successfully.")
print("Saved feature names path:", feature_names_path)

Feature names saved successfully.
Saved feature names path: c:\Users\User\OneDrive\Desktop\icu-early-warning-system\models\model_feature_names.joblib


In [12]:
model_metadata = {
    "model_name": "Gradient Boosting + SMOTE",
    "model_type": "GradientBoostingClassifier",
    "imbalance_method": "SMOTE",
    "target_column": target_col,
    "number_of_features": len(feature_names),
    "features": feature_names,
    "random_state": 42,
    "test_size": 0.25,
    "roc_auc_before_saving": auc_before
}

metadata_path = MODELS_DIR / "model_metadata.joblib"

joblib.dump(model_metadata, metadata_path)

print("Model metadata saved successfully.")
print("Saved metadata path:", metadata_path)

Model metadata saved successfully.
Saved metadata path: c:\Users\User\OneDrive\Desktop\icu-early-warning-system\models\model_metadata.joblib


In [13]:
loaded_model = joblib.load(model_path)
loaded_feature_names = joblib.load(feature_names_path)
loaded_metadata = joblib.load(metadata_path)

print("Saved model reloaded successfully.")
print("Loaded model:", loaded_metadata["model_name"])
print("Number of loaded features:", len(loaded_feature_names))

Saved model reloaded successfully.
Loaded model: Gradient Boosting + SMOTE
Number of loaded features: 16


In [14]:
y_pred_after = loaded_model.predict(X_test)
y_prob_after = loaded_model.predict_proba(X_test)[:, 1]

auc_after = roc_auc_score(y_test, y_prob_after)

print("ROC-AUC after loading:", round(auc_after, 3))

same_predictions = np.array_equal(y_pred_before, y_pred_after)
same_probabilities = np.allclose(y_prob_before, y_prob_after)

print("Same class predictions:", same_predictions)
print("Same probabilities:", same_probabilities)

ROC-AUC after loading: 0.758
Same class predictions: True
Same probabilities: True


In [15]:
persistence_results = pd.DataFrame({
    "Metric": [
        "ROC-AUC Before Saving",
        "ROC-AUC After Loading",
        "Same Class Predictions",
        "Same Prediction Probabilities"
    ],
    "Value": [
        auc_before,
        auc_after,
        same_predictions,
        same_probabilities
    ]
})

persistence_results

,Metric,Value
0,ROC-AUC Before Saving,0.757576
1,ROC-AUC After Loading,0.757576
2,Same Class Predictions,True
3,Same Prediction Probabilities,True


In [16]:
persistence_results.to_csv(
    RESULTS_DIR / "day10_model_persistence_validation.csv",
    index=False
)

print("Model persistence validation results saved successfully.")

Model persistence validation results saved successfully.


In [17]:
sample_patient = X_test.iloc[[0]]

sample_probability = loaded_model.predict_proba(sample_patient)[0, 1]
sample_prediction = loaded_model.predict(sample_patient)[0]

print("Sample patient prediction:")
print("Predicted sepsis probability:", round(sample_probability, 3))
print("Predicted class:", sample_prediction)

Sample patient prediction:
Predicted sepsis probability: 0.0
Predicted class: 0


## Clinical and Deployment Interpretation

Model persistence is a key step toward deployment.

By saving the trained Gradient Boosting + SMOTE model, the ICU Early Warning Prediction System can reuse the same model later without retraining.

This supports:

- dashboard development
- clinical risk scoring
- reproducible predictions
- future API deployment
- consistent patient-level inference

The validation step confirms that the loaded model produces the same predictions as the original model before saving.

In [18]:
print("Week 2 Day 3 completed successfully.")
print("Generated outputs:")
print("- gradient_boosting_smote_model.joblib")
print("- model_feature_names.joblib")
print("- model_metadata.joblib")
print("- day10_model_persistence_validation.csv")

Week 2 Day 3 completed successfully.
Generated outputs:
- gradient_boosting_smote_model.joblib
- model_feature_names.joblib
- model_metadata.joblib
- day10_model_persistence_validation.csv
